# Evaluate a trained `best.pt`

Confusion matrix, per-class PR curves, and an error gallery (worst false positives + false negatives). This is what tells you **what kind** of mistakes the model makes — which guides the negative-class fine-tune (Phase B5).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
MODEL = ROOT / 'models' / 'best.pt'
DATA = ROOT / 'data' / 'merged' / 'data.yaml'
assert MODEL.exists(), f'no model at {MODEL}'
assert DATA.exists(), f'no dataset at {DATA}'

In [ ]:
from ultralytics import YOLO
model = YOLO(str(MODEL))
metrics = model.val(data=str(DATA), split='test', imgsz=640, plots=True)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)
for i, name in enumerate(metrics.names.values() if hasattr(metrics, 'names') else ['fire', 'smoke']):
    if i < len(metrics.box.maps):
        print(f'  {name}: mAP50-95 = {metrics.box.maps[i]:.3f}')

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
save_dir = Path(metrics.save_dir)
for fname in ['confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    p = save_dir / fname
    if p.exists():
        plt.figure(figsize=(8, 6)); plt.imshow(Image.open(p)); plt.axis('off')
        plt.title(fname); plt.show()

## Error gallery: worst false positives and false negatives

Visual inspection beats summary metrics. Look for systematic mistakes: "all FP are sunsets," "all FN are dilute smoke." That's what site-tuning (Phase B5) addresses.

In [ ]:
import yaml
with open(DATA) as f:
    dy = yaml.safe_load(f)
TEST_IMG = ROOT / 'data' / 'merged' / dy.get('test', 'test/images')
TEST_LBL = TEST_IMG.parent / 'labels'
print('test images:', TEST_IMG)

In [ ]:
from PIL import Image
import torch
from collections import defaultdict

def parse_lbl(p):
    if not p.exists() or p.stat().st_size == 0:
        return set()
    out = set()
    with open(p) as f:
        for line in f:
            parts = line.strip().split()
            if parts and parts[0].lstrip('-').isdigit():
                out.add(int(parts[0]))
    return out

fps, fns = [], []
for img_p in list(TEST_IMG.iterdir())[:200]:  # first 200 for speed
    truth = parse_lbl(TEST_LBL / (img_p.stem + '.txt'))
    res = model.predict(str(img_p), conf=0.4, verbose=False)[0]
    pred = {int(b.cls) for b in res.boxes}
    if pred - truth:
        fps.append((img_p, sorted(pred - truth), sorted(truth)))
    if truth - pred:
        fns.append((img_p, sorted(truth - pred), sorted(pred)))
print(f'sampled false positives: {len(fps)}, false negatives: {len(fns)}')

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def grid(items, title, n=8):
    items = items[:n]
    if not items:
        print(f'{title}: none')
        return
    cols = 4; rows = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = list(axes.flatten()) if rows * cols > 1 else [axes]
    for ax, (p, pred, truth) in zip(axes, items):
        ax.imshow(Image.open(p))
        ax.set_title(f'pred={pred} truth={truth}', fontsize=8)
        ax.axis('off')
    for ax in axes[len(items):]:
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

grid(fps, 'False positives (predicted classes not in truth)')
grid(fns, 'False negatives (truth classes not predicted)')